In [7]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup as bs
from myfunc import text_clean

In [8]:
url="https://www.yes24.com/product/category/bestseller"
payload = dict(categoryNumber="001", pageNumber=1, pageSize=24)
r = requests.get(url, params = payload)
soup = bs(r.content, "lxml")
soup

<!DOCTYPE html>
<html lang="ko">
<head><link href="https://m.yes24.com/Home/Best?DispNo=001" media="only screen and(max-width: 640px)" rel="alternate"/>
<meta content="IE=Edge" http-equiv="X-UA-Compatible"/>
<meta content="text/html;charset=utf-8" http-equiv="Content-Type"/>
<meta content="dpr, width, viewport-width, rtt, downlink, ect, UA, UA-Platform, UA-Arch, UA-Model, UA-Mobile, UA-Full-Version" http-equiv="Accept-CH"/>
<meta content="86400" http-equiv="Accept-CH-Lifetime"/>
<meta content="unsafe-url" name="referrer"/>
<meta content="width=1170" name="viewport"/>
<title> 국내도서 종합 베스트 - 예스24 </title>
<meta content="예스24" name="title"/>
<meta content="도서, 티켓, 음반, 굿즈, 커뮤니티까지 문화 전반을 아우르는 국내 대표 문화콘텐츠 플랫폼, 아낌없는 혜택과 총알배송을 경험해보세요." name="description"/>
<meta content="인터넷 서점, 온라인 쇼핑, 상품 추천, 쇼핑몰, 상품 검색, 도서 정보, 국내도서, 외국도서, 전자책, eBook, 이북, 크레마, 공연, 콘서트, 뮤지컬, 음반, 예매, DVD, 블루레이, 예스24, YES24, 교보문고, 알라딘, 리센스, 예스24 도서용품, 친환경 PB 브랜드, 사은품, 굿즈" name="keywords"/>
<link href="//www.yes24.com/OpensearchDe

# 종류, 책 제목, 상세페이지 링크, 저자/역/글그림/그림, 출판사, 출간일, 가격, 회원리뷰수, 평점

In [10]:
result={}
for item in soup.select("#yesBestList > li"):
    
    book_id=item.select_one(".gd_name")["href"].split("/")[-1]

    # 책 종류
    kind=text_clean(item.select_one(".gd_res").get_text())

    # 책 제목
    book_title=item.select_one(".gd_name").get_text()

    # 상세페이지 링크
    detail_link="https://www.yes24.com" + item.select_one(".gd_name")["href"]

    # 저자/역자
    author=item.select_one(".authPub.info_auth").get_text().replace("\n", "").replace("\r", "")

    # 출판사
    publisher=item.select_one(".authPub.info_pub").get_text()

    # 출간일
    pub_date=item.select_one(".authPub.info_date").get_text()

    # 가격
    book_price=int(item.select_one("div.info_row.info_price em.yes_b").get_text().replace(",",""))

    # 리뷰수
    n_reviews=int(item.select_one(".txC_blue").get_text().replace(",","")) if item.select_one(".txC_blue") != None else 0

    # 리뷰 평점
    rating=float(item.select_one(".rating_grade em.yes_b").get_text()) if item.select_one(".rating_grade em.yes_b") != None else 0

    keys=("book_id", "kind","book_title","detail_link","author","publisher","pub_date","book_price","n_reviews","rating")
    
    values=(book_id, kind,book_title,detail_link,author,publisher,pub_date,book_price,n_reviews,rating)
    
    for key,value in zip(keys, values):
        result.setdefault(key, []).append(value)

df=pd.DataFrame(result)
df
    
    

,book_id,kind,book_title,detail_link,author,publisher,pub_date,book_price,n_reviews,rating
0,164510819,도서,괴테는 모든 것을 말했다,https://www.yes24.com/product/goods/164510819,스즈키 유이 저/이지수 역,리프,2025년 11월,15300,321,9.0
1,177044720,도서,인생을 위한 최소한의 생각,https://www.yes24.com/product/goods/177044720,"신영준, 고영성 저",상상스퀘어,2026년 02월,17820,31,9.2
2,180975910,도서,인생 마지막에 쓰는 주식투자 교과서,https://www.yes24.com/product/goods/180975910,정규준 저,애덤스미스,2026년 03월,22500,19,9.9
3,180137091,도서,완벽한 원시인,https://www.yes24.com/product/goods/180137091,자청 저,필로틱,2026년 03월,19800,72,9.4
4,173082175,도서,나의 완벽한 장례식,https://www.yes24.com/product/goods/173082175,조현선 저,북로망스,2026년 01월,17100,110,9.5
5,101375755,도서,프로젝트 헤일메리 (영화 특별판),https://www.yes24.com/product/goods/101375755,앤디 위어 저/강동혁 역,알에이치코리아(RHK),2026년 02월,19800,813,9.7
6,167466420,도서,"2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상",https://www.yes24.com/product/goods/167466420,최태성 저,이투스북,2025년 12월,17100,64,9.9
7,168226997,도서,진보를 위한 주식투자,https://www.yes24.com/product/goods/168226997,이광수 저,21세기북스,2025년 12월,19800,172,8.9
8,167466432,도서,"2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 하",https://www.yes24.com/product/goods/167466432,최태성 저,이투스북,2025년 12월,16650,46,10.0
9,149385533,도서,자몽살구클럽,https://www.yes24.com/product/goods/149385533,한로로 저,어센틱,2025년 07월,10800,213,9.2


# 저자 분리하기

In [11]:
author_result = []
for item in soup.select("#yesBestList > li"):
    result = {} # 1권의 저자 정보가 모임
    authors = item.select_one(".authPub.info_auth").get_text()
    if "보기/감추기" in authors:
        temp = authors.split("보기/감추기")
        author = temp[1].strip().split("\n")
        result['저'] = author
#         print(author)
    else:
        for name_by_role in item.select_one(".authPub.info_auth").get_text().split("/"):
            temp = name_by_role.split()
            name = " ".join(temp[:-1])
            role = temp[-1]
            result[role] = name
    
    author_result.append(result)    
df = pd.DataFrame(author_result)
df

,저,역,글,그림,글그림
0,스즈키 유이,이지수,NaN,NaN,NaN
1,"신영준, 고영성",NaN,NaN,NaN,NaN
2,정규준,NaN,NaN,NaN,NaN
3,자청,NaN,NaN,NaN,NaN
4,조현선,NaN,NaN,NaN,NaN
5,앤디 위어,강동혁,NaN,NaN,NaN
6,최태성,NaN,NaN,NaN,NaN
7,이광수,NaN,NaN,NaN,NaN
8,최태성,NaN,NaN,NaN,NaN
9,한로로,NaN,NaN,NaN,NaN


# yes24 종합베스트 999권 자료 수집해서 csv 파일로 저장하기
* book_data/
* yes24_수집일_best.csv
* 수집일은 datetime에서 오늘 날짜 추출해서 파일명에 포함시키기

In [13]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup as bs
from myfunc import text_clean

In [16]:
from datetime import datetime
datetime.now()
now=datetime.now()
print(f"{now.year}/{now.month}/{now.day}")

2026/3/16


In [17]:
today=datetime.today()
print(today)

2026-03-16 15:38:25.681392


In [18]:
today = f"{today.year}{today.month:02d}{today.day:02d}"
today

'20260316'

In [19]:
dir_name="book_data"
if not os.path.exists(dir_name):
    os.mkdir(dir_name)

In [20]:
result={}

for num in range(1,43):
    print(f"{num}/{42}    수집중", end="\r")
    url="https://www.yes24.com/product/category/bestseller"
    payload = dict(categoryNumber="001", pageNumber=num, pageSize=24)
    r = requests.get(url, params = payload)
    soup = bs(r.content, "lxml")

    
    for item in soup.select("#yesBestList > li"):

        book_id=item.select_one(".gd_name")["href"].split("/")[-1]

        # 책 종류
        kind=text_clean(item.select_one(".gd_res").get_text())

        # 책 제목
        book_title=item.select_one(".gd_name").get_text()

        # 상세페이지 링크
        detail_link="https://www.yes24.com" + item.select_one(".gd_name")["href"]

        # 저자/역자
        author=item.select_one(".authPub.info_auth").get_text().replace("\n", "").replace("\r", "")

        # 출판사
        publisher=item.select_one(".authPub.info_pub").get_text()

        # 출간일
        pub_date=item.select_one(".authPub.info_date").get_text()

        # 가격
        book_price=int(item.select_one("div.info_row.info_price em.yes_b").get_text().replace(",",""))

        # 리뷰수
        n_reviews=int(item.select_one(".txC_blue").get_text().replace(",","")) if item.select_one(".txC_blue") != None else 0

        # 리뷰 평점
        rating=float(item.select_one(".rating_grade em.yes_b").get_text()) if item.select_one(".rating_grade em.yes_b") != None else 0

        keys=("book_id", "kind","book_title","detail_link","author","publisher","pub_date","book_price","n_reviews","rating")

        values=(book_id, kind,book_title,detail_link,author,publisher,pub_date,book_price,n_reviews,rating)

        for key,value in zip(keys, values):
            result.setdefault(key, []).append(value)
        
#         time.sleep(3)
        
dir_name="book_data"
if not os.path.exists(dir_name):
    os.mkdir(dir_name)

df=pd.DataFrame(result)
df

df.to_csv(f"./book_data/yes24_{now.year}.{now.month}.{now.day}_best.csv", encoding="utf-8-sig")

# yes24_수집일_best.csv 파일에서 상세링크만 가져와서 상세페이지에 방문하고 
* book_id
* 쪽수, 무게, 크기 (다 따로)
* 카테고리 분류 (묶어도 상관없음)
* yes_24 수집일_best_detail.csv

# 수집하기

In [21]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup as bs
from myfunc import text_clean

In [22]:
from datetime import datetime

In [23]:
today = datetime.today()
print(today)

2026-03-16 15:39:17.817235


In [24]:
today = f"{today.year}{today.month:02d}{today.day:02d}"
today

'20260316'

In [27]:
data = pd.read_csv("./book_data/yes24_20260316_best.csv")
detail_links = data['detail_link']
detail_links

FileNotFoundError: [Errno 2] No such file or directory: './book_data/yes24_20260316_best.csv'

In [ ]:
all_detail_info = []
for idx, url in enumerate(detail_links):
    print(f"{idx}/{len(detail_links)}   수집중", end="\r")
    r = requests.get(url)
    soup = bs(r.content, "lxml")
    
    book_info ={}
    
    # book_id
    book_id = url.split("/")[-1]
    book_info['book_id'] = book_id 
    
    # 쪽수, 무게, 크기
    for row in soup.select("tbody.b_size tr"):
        if "쪽수" in row.select_one("th").get_text():
            pummoc_info = row.select_one("td").get_text().split("|")
    
    print(pummoc_info)
    for item in pummoc_info:
        temp = item.strip()
        if temp[-1] == "쪽":
            book_info["페이지"] = temp
        elif temp[-1] == "g":
            book_info['무게'] = temp
        elif temp[-1] == "m":
            book_info['크기'] = temp

    #카테고리 분류
    category = soup.select_one("#infoset_goodsCate .yesAlertLi > li").get_text(strip=True).split(">")    
    book_info['카테고리'] = category
   
    all_detail_info.append(book_info)
    
#     time.sleep(3)

df2 = pd.DataFrame(all_detail_info)
df2

In [ ]:
df2 = pd.DataFrame(all_detail_info)

dir_name = "book_data"
if not os.path.exists(dir_name):
        os.mkdir(dir_name)
        print(f"{dir_name}을 생성했습니다.")        
        

df2.to_csv(f"./{dir_name}/yes24_{today}_best_detail.csv")
df2